# <span style="color: #1F1DB5;">SPRINT 3 – Manipulación de datos (Data Wrangling)

# <span style="color: #1F1DB5;">Notebook 2 – Preprocesamiento de Datos — Versión estudiante

## Cómo usar esta versión estudiante

Este notebook está preparado para que tomes apuntes, completes ejercicios y documentes tus decisiones durante la clase.

**Modo de trabajo recomendado:**

1. Lee primero la consigna de cada bloque o ejercicio.
2. Completa los espacios de respuesta antes de comparar con la explicación del instructor/a.
3. Ejecuta las celdas de setup cuando existan.
4. Escribe interpretación, dudas y decisiones en los espacios de apuntes.
5. Al final, revisa si tus respuestas conectan datos, método e implicación de negocio.

> Las salidas de ejecución fueron limpiadas para que puedas correr el notebook desde cero.


## Notas de clase principales

- Identificar y manejar valores faltantes (NaN) con distintas estrategias.
- Detectar y tratar valores atípicos (outliers) de forma fundamentada.
- Aplicar técnicas de transformación de datos: normalización, estandarización, encoding.
- Comprender el impacto del preprocesamiento en la calidad de los modelos.
- Construir un pipeline básico de limpieza reutilizable.

### Mis notas iniciales

- 
- 
- 


## <span style="color: #2826DE;">Objetivos de Aprendizaje

- Identificar y manejar **valores faltantes** (NaN) con distintas estrategias
- Detectar y tratar **valores atípicos (outliers)** de forma fundamentada
- Aplicar técnicas de **transformación de datos**: normalización, estandarización, encoding
- Comprender el impacto del preprocesamiento en la calidad de los modelos
- Construir un **pipeline básico de limpieza** reutilizable

### <span style="color: #2772F2;">Agenda (2 horas)

| Tema | Duración |
|---|---|
Introducción + Pregunta guía | 10 min |
Valores faltantes: detección y estrategias | 25 min |
Outliers: detección y tratamiento | 20 min |
Transformaciones: normalización y encoding | 20 min |
Actividad práctica (Breakout Rooms) | 20 min |
Tips y buenas prácticas | 5 min |
Cierre y Kahoot | 5 min |

## <span style="color: #2826DE;">❓ Pregunta Guía

### ¿Por qué decimos que los **datos sucios** generan **conclusiones peligrosas**?

### Mi respuesta inicial

- 


<img src="../../Images/S3_DS_dirty_data.png" width="600">

## <span style="color: #2826DE;">Valores faltantes: detección y estrategias (25 mins)

Antes de limpiar datos, necesitamos entender cuantos problemas tenemos. Vamos a cargar el dataset de pinguinos, que tiene valores nulos reales, y diagnosticar el estado de los datos. Siempre empieza con un diagnostico antes de tomar decisiones de limpieza.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset con valores faltantes
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Preguntas:

- ¿Cuántas filas tiene el dataset? Cuántas columnas?

- ¿Qué columna tiene mas valores nulos? ¿Qué porcentaje representa?

- ¿Por qué crees que hay valores nulos en este dataset de pinguinos?

### Mis respuestas de validación

- 
- 


In [ ]:
# Detectar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nPorcentaje de nulos:")
print((df.isnull().sum() / len(df) * 100).round(2))

### <span style="color: #2772F2;">Estrategias para manejar valores nulos

| Estrategia | Cuándo usarla | Código |
|---|---|---|
| **Eliminar filas** | Pocos nulos, distribuidos aleatoriamente | `df.dropna()` |
| **Rellenar con media** | Variables numéricas, distribución simétrica | `df.fillna(df.mean())` |
| **Rellenar con mediana** | Variables numéricas con outliers | `df.fillna(df.median())` |
| **Rellenar con moda** | Variables categóricas | `df.fillna(df.mode()[0])` |
| **Rellenar con "Unknown"** | Categóricas donde el nulo tiene significado | `df.fillna("Unknown")` |
| **Interpolación** | Series temporales | `df.interpolate()` |

⚠️ **Nunca** reemplaces nulos numéricos con 0 sin pensar — puede distorsionar estadísticas.

Antes de rellenar los nulos, necesitamos decidir la estrategia correcta para cada columna. No existe una sola respuesta correcta: depende del tipo de dato, la distribucion y el contexto del problema. La tabla de arriba es tu guia de referencia.

La eleccion de estrategia para manejar nulos es una decision de negocio, no solo tecnica. Rellenar con la media puede distorsionar la distribucion si hay outliers. Eliminar filas puede sesgar el análisis si los nulos no son aleatorios. Siempre documenta tu decision y el razonamiento detras de ella.

In [ ]:
df_limpio = df.copy()

# Rellenar numéricas con mediana
for col in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].median())

# Rellenar categóricas con moda
df_limpio["sex"] = df_limpio["sex"].fillna(df_limpio["sex"].mode()[0])

print("Nulos restantes:", df_limpio.isnull().sum().sum())
df_limpio.head()

Preguntas:

- ¿Por qué usamos la mediana para las columnas numericas y no la media?

- ¿Por qué usamos la moda para la columna sex?

- ¿Qué pasaría si rellenaramos todos los nulos con 0?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Outliers: detección y tratamiento (20 mins)

### <span style="color: #2772F2;">Método IQR (Rango Intercuartílico)

Los bigotes de un boxplot se calculan así:
- **Límite inferior** = Q1 − 1.5 × IQR
- **Límite superior** = Q3 + 1.5 × IQR

Cualquier valor fuera de estos límites es un **outlier teórico**.

El metodo IQR (Rango Intercuartilico) es el estandar para detectar outliers. La idea es simple: calculamos donde estan el 25% y el 75% de los datos, y cualquier valor que este muy lejos de ese rango central se considera un outlier. Los bigotes del boxplot marcan exactamente estos limites.

In [ ]:
# Detectar outliers con IQR
col = "body_mass_g"

Q1 = df_limpio[col].quantile(0.25)
Q3 = df_limpio[col].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Límite inferior: {limite_inferior}")
print(f"Límite superior: {limite_superior}")

outliers = df_limpio[(df_limpio[col] < limite_inferior) | (df_limpio[col] > limite_superior)]
print(f"\nOutliers encontrados: {len(outliers)}")
print(outliers[["species", col]])

Preguntas:

- ¿Cuántos outliers encontraste? ¿Son muchos o pocos en relación al total?

- ¿Deberíamos eliminar estos outliers? ¿Por qué si o por qué no?

- ¿Qué tipo de modelo sería más sensible a estos outliers?

### Mis respuestas de validación

- 
- 


In [ ]:
# Visualizar con boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot por especie
sns.boxplot(x="species", y="body_mass_g", data=df_limpio, ax=axes[0])
axes[0].set_title("Masa corporal por especie (Boxplot)")
axes[0].axhline(y=limite_superior, color='red', linestyle='--', label='Límite superior')
axes[0].axhline(y=limite_inferior, color='orange', linestyle='--', label='Límite inferior')
axes[0].legend()

# Histograma
sns.histplot(df_limpio["body_mass_g"], kde=True, ax=axes[1])
axes[1].set_title("Distribución de masa corporal")
axes[1].axvline(x=limite_superior, color='red', linestyle='--')
axes[1].axvline(x=limite_inferior, color='orange', linestyle='--')

plt.tight_layout()
plt.show()

Preguntas:

- ¿Qué informacion adicional te da el histograma que no te da el boxplot?

- ¿Si un pinguino pesa 7000g, lo eliminarías del dataset? ¿Por qué?

### Mis respuestas de validación

- 
- 


### <span style="color: #2772F2;">¿Eliminar o conservar outliers?

| Situación | Decisión |
|---|---|
| Error de medición o captura | Eliminar |
| Valor real pero extremo | Conservar (puede ser información valiosa) |
| Modelo sensible a outliers (regresión lineal) | Eliminar o transformar |
| Modelo robusto (árboles de decisión) | Conservar |

> **Siempre documenta** por qué eliminaste o conservaste un outlier.

## <span style="color: #2826DE;">Transformaciones: normalización y encoding (20 mins)

<img src="../../Images/S3_DS_OHE.png" width="600">

Las transformaciones preparan los datos para los modelos de Machine Learning. Los modelos como regresion lineal y redes neuronales son sensibles a la escala de los datos. La normalizacion y estandarizacion resuelven este problema. El One-Hot Encoding convierte categorias en numeros que los modelos pueden procesar.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Normalización (MinMax): escala entre 0 y 1
scaler_minmax = MinMaxScaler()
df_limpio["body_mass_normalizado"] = scaler_minmax.fit_transform(df_limpio[["body_mass_g"]])

# Estandarización (Z-score): media=0, std=1
scaler_std = StandardScaler()
df_limpio["body_mass_estandarizado"] = scaler_std.fit_transform(df_limpio[["body_mass_g"]])

print(df_limpio[["body_mass_g", "body_mass_normalizado", "body_mass_estandarizado"]].head(8))

Preguntas:

- ¿Cuál es la diferencia entre normalización (MinMax) y estandarización (Z-score)?

- ¿Cuándo usarias normalizacion y cuando estandarización?

- ¿Por qué el valor normalizado de la masa mínima es 0 y el máximo es 1?

### Mis respuestas de validación

- 
- 


In [ ]:
# One-Hot Encoding para variables categóricas
df_encoded = pd.get_dummies(df_limpio[["species", "island", "sex"]], drop_first=False)
print("Columnas después de encoding:")
print(df_encoded.columns.tolist())
df_encoded.head()

Preguntas:

- ¿Por qué necesitamos convertir las categorias en numeros para los modelos de ML?

- ¿Cuántas columnas nuevas se crean con One-Hot Encoding para la columna species?

- ¿Qué problema podría causar tener demasiadas columnas después del encoding?

### Mis respuestas de validación

- 
- 


## <span style="color: #2826DE;">Actividad práctica – Breakout Rooms (20 mins)

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


### <span style="color: #aa0808;">En esta ocasión, quien compartirá resultados será: (Giremos la ruleta)

### Ejercicio – Pipeline de limpieza completo

Usando el dataset de pingüinos:

1. Detecta y reporta todos los valores nulos
2. Aplica la estrategia apropiada para cada columna
3. Detecta outliers en `flipper_length_mm` usando IQR
4. Visualiza los outliers con un boxplot
5. Normaliza `bill_length_mm` entre 0 y 1
6. Aplica One-Hot Encoding a `species`
7. Guarda el dataset limpio

### Espacio de trabajo del estudiante

**Respuesta, decisiones o interpretación:**

- 
- 


In [ ]:
# TODO estudiante: completa el código de acuerdo con la consigna anterior.
# Contexto: Tu pipeline de limpieza aquí


## <span style="color: #2826DE;">Tips y buenas prácticas (5 mins)

- **Siempre trabaja con copias**: `df_limpio = df.copy()`
- **Documenta cada decisión** de limpieza con comentarios
- **Verifica antes y después** con `.info()` y `.describe()`
- **No elimines outliers automáticamente** — investiga primero
- **Guarda el dataset limpio** para reproducibilidad

## <span style="color: #2826DE;">Cierre y Kahoot (5 mins)

Kahoot - Preguntas sugeridas

1️⃣  ¿Qué hace df.isnull().sum()?

- Elimina todos los valores nulos del DataFrame
- Rellena los nulos con cero
- Muestra las primeras filas del DataFrame
- Cuenta los valores nulos por columna  


2️⃣ ¿Cuándo usar la mediana en lugar de la media para rellenar nulos?

- Siempre, la mediana es siempre mejor
- Cuando la columna tiene datos categoricos
- Cuando todos los valores son iguales
- Cuando hay outliers que distorsionarian la media  


3️⃣ ¿Qué es el IQR?

- El promedio de todos los valores
- La diferencia entre el maximo y el minimo
- Un tipo de grafico estadistico
- La diferencia entre el tercer y primer cuartil, Q3 - Q1 


4️⃣ ¿Qué hace MinMaxScaler?

- Elimina los outliers del dataset
- Convierte variables categoricas en numericas
- Calcula la media y desviacion estandar
- Escala los valores entre 0 y 1  


5️⃣  ¿Para qué sirve One-Hot Encoding?

- Para normalizar variables numericas
- Para eliminar valores nulos
- Para detectar outliers
- Para convertir categorias en columnas binarias  


6️⃣ ¿Por qué siempre usar df.copy() antes de limpiar datos?

- Para hacer el codigo mas rapido
- Para ahorrar memoria RAM
- Para crear un backup automatico en disco
- Para no modificar el dataset original  


7️⃣ ¿Qué estrategia usar si el 60% de una columna son nulos?

- Rellenar con la mediana
- Rellenar con la moda
- Rellenar con cero
- Considerar eliminar la columna completa  


8️⃣ ¿Qué hace df.drop_duplicates()?

- Elimina columnas duplicadas
- Elimina filas con valores nulos
- Ordena el DataFrame por valores duplicados
- Elimina filas completamente duplicadas  